In [15]:
from pathlib import Path
from datetime import datetime
import pandas as pd
from fitparse import FitFile

import xml.etree.ElementTree as ET

base_dir = Path("../../Strava-Export/activities")
if not base_dir.exists():
    raise FileNotFoundError(f"Path not found: {base_dir.resolve()}")

# Optional FIT parser (if installed)
try:
    HAS_FITPARSE = True
except Exception:
    HAS_FITPARSE = False

files = sorted(
    [p for p in base_dir.rglob("*") if p.is_file() and p.suffix.lower() in {".gpx", ".fit"}]
)

def parse_gpx_info(path: Path):
    """Extract lightweight metadata from GPX without external deps."""
    info = {
        "start_time": pd.NaT,
        "end_time": pd.NaT,
        "trackpoints": pd.NA,
    }
    try:
        root = ET.parse(path).getroot()
        # Handle namespaces
        ns = {"g": root.tag.split("}")[0].strip("{")} if "}" in root.tag else {}
        trkpts = root.findall(".//g:trkpt", ns) if ns else root.findall(".//trkpt")
        times = root.findall(".//g:time", ns) if ns else root.findall(".//time")

        info["trackpoints"] = len(trkpts)
        if times:
            parsed = pd.to_datetime([t.text for t in times if t.text], errors="coerce", utc=True).dropna()
            if len(parsed) > 0:
                info["start_time"] = parsed.min()
                info["end_time"] = parsed.max()
    except Exception:
        pass
    return info

def parse_fit_info(path: Path):
    """Extract basic FIT metadata if fitparse is available."""
    info = {
        "start_time": pd.NaT,
        "end_time": pd.NaT,
        "distance_km": pd.NA,
        "duration_s": pd.NA,
    }
    if not HAS_FITPARSE:
        return info

    try:
        fit = FitFile(path)
        session_msgs = list(fit.get_messages("session"))
        if session_msgs:
            fields = {f.name: f.value for f in session_msgs[0]}
            st = fields.get("start_time")
            tt = fields.get("total_timer_time")
            dist = fields.get("total_distance")

            if st is not None:
                info["start_time"] = pd.to_datetime(st, utc=True, errors="coerce")
            if tt is not None and pd.notna(tt):
                info["duration_s"] = float(tt)
                if pd.notna(info["start_time"]):
                    info["end_time"] = info["start_time"] + pd.to_timedelta(info["duration_s"], unit="s")
            if dist is not None and pd.notna(dist):
                info["distance_km"] = float(dist) / 1000.0
    except Exception:
        pass
    return info

rows = []
for p in files:
    row = {
        "file": str(p.relative_to(base_dir)),
        "ext": p.suffix.lower(),
        "size_mb": p.stat().st_size / (1024 * 1024),
        "modified": pd.to_datetime(datetime.fromtimestamp(p.stat().st_mtime), utc=True),
    }
    if p.suffix.lower() == ".gpx":
        row.update(parse_gpx_info(p))
    elif p.suffix.lower() == ".fit":
        row.update(parse_fit_info(p))
    rows.append(row)

df = pd.DataFrame(rows)

print(f"Base directory: {base_dir.resolve()}")
print(f"Total .gpx/.fit files: {len(df)}")
print(f"Total size: {df['size_mb'].sum():.2f} MB")
print("\nCounts by extension:")
print(df["ext"].value_counts(dropna=False))

if not df.empty:
    # Time coverage from available activity timestamps
    time_min = pd.to_datetime(df["start_time"], errors="coerce", utc=True).min()
    time_max = pd.to_datetime(df["end_time"], errors="coerce", utc=True).max()
    print(f"\nTime coverage (from parsed activity times): {time_min} -> {time_max}")

    # Overview table
    display_cols = ["file", "ext", "size_mb", "start_time", "end_time", "trackpoints", "distance_km", "duration_s", "modified"]
    for c in display_cols:
        if c not in df.columns:
            df[c] = pd.NA
    display(df[display_cols].sort_values(["ext", "start_time", "modified"], na_position="last").head(30))

    # Annual summary if timestamps exist
    ts = pd.to_datetime(df["start_time"], errors="coerce", utc=True)
    yearly = (
        df.assign(year=ts.dt.year)
          .dropna(subset=["year"])
          .groupby(["year", "ext"], dropna=False)
          .agg(
              files=("file", "count"),
              total_size_mb=("size_mb", "sum"),
              total_distance_km=("distance_km", "sum"),
              total_duration_h=("duration_s", lambda s: pd.to_numeric(s, errors="coerce").sum() / 3600),
          )
          .reset_index()
          .sort_values(["year", "ext"])
    )
    if not yearly.empty:
        print("\nYearly summary:")
        display(yearly)

if not HAS_FITPARSE and (df["ext"] == ".fit").any():
    print("\nNote: 'fitparse' is not installed, so FIT metadata extraction is limited.")
    print("Install with: pip install fitparse")

Base directory: /Users/as/Library/Mobile Documents/com~apple~CloudDocs/Dokumente/Studium/Master-26-FS/Big Data Methods for Economists/Strava-Export/activities
Total .gpx/.fit files: 73
Total size: 33.03 MB

Counts by extension:
ext
.gpx    61
.fit    12
Name: count, dtype: int64

Time coverage (from parsed activity times): 2023-07-06 09:47:55+00:00 -> 2024-03-29 11:47:41+00:00


,file,ext,size_mb,start_time,end_time,trackpoints,distance_km,duration_s,modified
34,8993747317.fit,.fit,0.183357,NaT,NaT,NaN,<NA>,<NA>,2026-03-22 20:13:34+00:00
35,9270819398.fit,.fit,0.230167,NaT,NaT,NaN,<NA>,<NA>,2026-03-22 20:13:38+00:00
36,9270913122.fit,.fit,0.089821,NaT,NaT,NaN,<NA>,<NA>,2026-03-22 20:13:38+00:00
37,9270922700.fit,.fit,0.190369,NaT,NaT,NaN,<NA>,<NA>,2026-03-22 20:13:38+00:00
38,9285478107.fit,.fit,1.401915,NaT,NaT,NaN,<NA>,<NA>,2026-03-22 20:14:10+00:00
39,9356662399.fit,.fit,0.208770,NaT,NaT,NaN,<NA>,<NA>,2026-03-22 20:14:10+00:00
40,9377072379.fit,.fit,0.173002,NaT,NaT,NaN,<NA>,<NA>,2026-03-22 20:14:10+00:00
42,9456089915.fit,.fit,0.268194,NaT,NaT,NaN,<NA>,<NA>,2026-03-22 20:14:10+00:00
31,12284151651.fit,.fit,0.324514,NaT,NaT,NaN,<NA>,<NA>,2026-03-22 20:14:50+00:00
32,12523846686.fit,.fit,0.096100,NaT,NaT,NaN,<NA>,<NA>,2026-03-22 20:14:54+00:00



Yearly summary:


,year,ext,files,total_size_mb,total_distance_km,total_duration_h
0,2023.0,.gpx,60,29.232556,0,0.0
1,2024.0,.gpx,1,0.494793,0,0.0


In [7]:
import pandas as pd

df = pd.read_csv("activity_df_sample.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 171516 entries, 0 to 171515
Data columns (total 40 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   source_file           171516 non-null  str    
 1   source_format         171516 non-null  str    
 2   altitude              57022 non-null   float64
 3   cadence               56334 non-null   float64
 4   distance              71342 non-null   float64
 5   enhanced_altitude     91411 non-null   float64
 6   enhanced_speed        91408 non-null   float64
 7   position_lat          168287 non-null  float64
 8   position_long         168287 non-null  float64
 9   speed                 82196 non-null   float64
 10  timestamp             94630 non-null   str    
 11  gps_accuracy          38732 non-null   float64
 12  heart_rate            25362 non-null   float64
 13  activity_type         6774 non-null    str    
 14  fractional_cadence    20331 non-null   float64
 15  power      

/var/folders/q9/d95wjtz97h5dptsf6gjsbvdc0000gn/T/ipykernel_49668/19846230.py:3: DtypeWarning: Columns (0: timestamp, 1: activity_type, 2: time) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("activity_df_sample.csv")


In [14]:
df["source_file"]

0            activity_files/activities/9456089915.fit
1            activity_files/activities/9456089915.fit
2            activity_files/activities/9456089915.fit
3            activity_files/activities/9456089915.fit
4            activity_files/activities/9456089915.fit
                             ...                     
171511    activity_files/activities/9273286393.gpx.gz
171512    activity_files/activities/9273286393.gpx.gz
171513    activity_files/activities/9273286393.gpx.gz
171514    activity_files/activities/9273286393.gpx.gz
171515    activity_files/activities/9273286393.gpx.gz
Name: source_file, Length: 171516, dtype: str